In [5]:
import pandas as pd
import numpy as np
from google.colab import files
uploaded = files.upload()
filename = next(iter(uploaded))
df = pd.read_csv(filename)

print("CSV loaded successfully!")

Saving HotelIQ_Data_Analysis.ipynb to HotelIQ_Data_Analysis (1).ipynb


ParserError: Error tokenizing data. C error: Expected 2 fields in line 45, saw 3


In [ ]:

# Show the first five bookings
display(df.head())

# Check the number of rows and columns
print("Dataset size:", df.shape)

# List all column names
print("\nColumn names:")
print(df.columns.tolist())

# Check data types and missing values
print("\nDataset information:")
df.info()

# Count missing values in each column
print("\nMissing values:")
print(df.isnull().sum().sort_values(ascending=False).head(15))

# Check for duplicate rows
print("\nDuplicate rows:", df.duplicated().sum())

In [ ]:
# Make a copy so the original data remains unchanged
df_original = df.copy()

# Count exact duplicate rows
duplicate_count = df_original.duplicated().sum()
print("Exact duplicate rows:", duplicate_count)

# Create a version with exact duplicates removed
df_deduplicated = df_original.drop_duplicates().copy()

print("Original rows:", len(df_original))
print("Rows after removing exact duplicates:", len(df_deduplicated))
print("Rows removed:", len(df_original) - len(df_deduplicated))

# Compare cancellation rates in both versions
print("\nCancellation rate — original:")
print(round(df_original["is_canceled"].mean() * 100, 2), "%")

print("\nCancellation rate — deduplicated:")
print(round(df_deduplicated["is_canceled"].mean() * 100, 2), "%")

In [ ]:
for label, data in [
    ("Original", df_original),
    ("Deduplicated", df_deduplicated)
]:
    print(f"\n{label} dataset")

    summary = (
        data.groupby("hotel")
        .agg(
            total_bookings=("is_canceled", "size"),
            canceled_bookings=("is_canceled", "sum"),
            cancellation_rate=("is_canceled", "mean")
        )
    )

    summary["cancellation_rate"] = (
        summary["cancellation_rate"] * 100
    ).round(2)

    display(summary)

In [ ]:

# Inspect a few rows that have exact duplicates
duplicate_rows = df_original[
    df_original.duplicated(keep=False)
].sort_values(
    by=["hotel", "arrival_date_year",
        "arrival_date_month", "arrival_date_day_of_month"]
)

display(duplicate_rows.head(10))


In [ ]:

# Create a working copy of the original dataset
df_clean = df_original.copy()

print("Rows:", len(df_clean))
print("Columns:", len(df_clean.columns))


In [ ]:

# Fill missing categorical information
df_clean["country"] = df_clean["country"].fillna("Unknown")

# Preserve missing IDs as categorical labels
df_clean["agent"] = df_clean["agent"].astype("Int64").astype("string").fillna("Unknown")
df_clean["company"] = df_clean["company"].astype("Int64").astype("string").fillna("Unknown")

# Fill the four missing children values with the median
children_median = df_clean["children"].median()
df_clean["children"] = df_clean["children"].fillna(children_median).astype(int)

# Check remaining missing values
print("Missing values remaining:")
print(df_clean.isnull().sum().sum())

In [ ]:
# Total number of nights booked
df_clean["total_nights"] = (
    df_clean["stays_in_weekend_nights"]
    + df_clean["stays_in_week_nights"]
)

# Total guests
df_clean["total_guests"] = (
    df_clean["adults"]
    + df_clean["children"]
    + df_clean["babies"]
)

# Create arrival date
df_clean["arrival_date"] = pd.to_datetime(
    df_clean["arrival_date_year"].astype(str) + "-"
    + df_clean["arrival_date_month"] + "-"
    + df_clean["arrival_date_day_of_month"].astype(str),
    errors="coerce"
)

# Convert reservation status date
df_clean["reservation_status_date"] = pd.to_datetime(
    df_clean["reservation_status_date"],
    errors="coerce"
)

# Estimated booking value (not actual realized revenue)
df_clean["estimated_booking_value"] = (
    df_clean["adr"] * df_clean["total_nights"]
)

display(df_clean[
    ["hotel", "arrival_date", "total_nights",
     "total_guests", "adr", "estimated_booking_value"]
].head())


In [ ]:

print("Zero-night bookings:",
      (df_clean["total_nights"] == 0).sum())

print("Zero-guest bookings:",
      (df_clean["total_guests"] == 0).sum())

print("Negative ADR:",
      (df_clean["adr"] < 0).sum())

print("Missing arrival dates:",
      df_clean["arrival_date"].isna().sum())

print("Missing reservation status dates:",
      df_clean["reservation_status_date"].isna().sum())


In [ ]:

# Export cleaned data
df_clean.to_csv("HotelIQ_Cleaned.csv", index=False)

print("Cleaned dataset saved successfully!")

# Download to your computer
from google.colab import files
files.download("HotelIQ_Cleaned.csv")

In [ ]:

# STEP 7 - Data Quality Validation

# Start from the cleaned dataset
df_master = df_clean.copy()

# Recalculate these columns if necessary
df_master["total_nights"] = (
    df_master["stays_in_weekend_nights"]
    + df_master["stays_in_week_nights"]
)

df_master["total_guests"] = (
    df_master["adults"]
    + df_master["children"]
    + df_master["babies"]
)

# Create the data quality flag
df_master["data_quality_issue"] = (
    (df_master["total_nights"] == 0)
    | (df_master["total_guests"] == 0)
    | (df_master["adr"] < 0)
)

# Create the analysis dataset
df_analysis = df_master.loc[
    ~df_master["data_quality_issue"]
].copy()

# Compare the datasets
print("Master dataset:", len(df_master))
print("Analysis dataset:", len(df_analysis))
print("Excluded records:", len(df_master) - len(df_analysis))

print(
    "Master cancellation rate:",
    round(df_master["is_canceled"].mean() * 100, 2), "%"
)

print(
    "Analysis cancellation rate:",
    round(df_analysis["is_canceled"].mean() * 100, 2), "%"
)


In [ ]:
import matplotlib.pyplot as plt

# Summarize hotel performance
hotel_performance = df_analysis.groupby("hotel").agg(
    total_bookings=("is_canceled", "size"),
    canceled_bookings=("is_canceled", "sum"),
    cancellation_rate=("is_canceled", "mean"),
    average_adr=("adr", "mean"),
    average_nights=("total_nights", "mean")
).reset_index()

hotel_performance["cancellation_rate"] *= 100

display(hotel_performance.round(2))

# Visualize cancellation rates
plt.figure(figsize=(8, 5))

plt.bar(
    hotel_performance["hotel"],
    hotel_performance["cancellation_rate"]
)

plt.title("HotelIQ - Cancellation Rate by Hotel")
plt.xlabel("Hotel Type")
plt.ylabel("Cancellation Rate (%)")

plt.tight_layout()
plt.show()

In [ ]:

df_master.to_csv("HotelIQ_Master.csv", index=False)
df_analysis.to_csv("HotelIQ_Analysis.csv", index=False)

print("Both datasets saved successfully!")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Create monthly cancellation summary
monthly_analysis = (
    df_analysis.groupby(
        ["arrival_date_month", "hotel"]
    )
    .agg(
        total_bookings=("is_canceled", "size"),
        canceled_bookings=("is_canceled", "sum"),
        cancellation_rate=("is_canceled", "mean")
    )
    .reset_index()
)

# Convert cancellation rate to percentage
monthly_analysis["cancellation_rate"] *= 100

# Arrange months chronologically
month_order = [
    "January", "February", "March", "April",
    "May", "June", "July", "August",
    "September", "October", "November", "December"
]

monthly_analysis["arrival_date_month"] = pd.Categorical(
    monthly_analysis["arrival_date_month"],
    categories=month_order,
    ordered=True
)

monthly_analysis = monthly_analysis.sort_values(
    ["arrival_date_month", "hotel"]
)

# Plot monthly cancellation rates
pivot_monthly = monthly_analysis.pivot(
    index="arrival_date_month",
    columns="hotel",
    values="cancellation_rate"
)

pivot_monthly.plot(
    kind="line",
    marker="o",
    figsize=(12, 6)
)

plt.title("HotelIQ - Monthly Cancellation Rates")
plt.xlabel("Arrival Month")
plt.ylabel("Cancellation Rate (%)")
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

display(monthly_analysis.round(2))


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Group bookings by how far in advance they were made
bins = [-1, 7, 30, 90, 180, float("inf")]

labels = [
    "0-7 days",
    "8-30 days",
    "31-90 days",
    "91-180 days",
    "181+ days"
]

df_analysis["lead_time_group"] = pd.cut(
    df_analysis["lead_time"],
    bins=bins,
    labels=labels
)

# Calculate booking count and cancellation rate
lead_time_analysis = (
    df_analysis.groupby(
        ["lead_time_group", "hotel"],
        observed=True
    )
    .agg(
        total_bookings=("is_canceled", "size"),
        canceled_bookings=("is_canceled", "sum"),
        cancellation_rate=("is_canceled", "mean")
    )
    .reset_index()
)

lead_time_analysis["cancellation_rate"] *= 100

display(lead_time_analysis.round(2))

# Visualize cancellation rates
pivot_lead = lead_time_analysis.pivot(
    index="lead_time_group",
    columns="hotel",
    values="cancellation_rate"
)

pivot_lead.plot(
    kind="bar",
    figsize=(11, 6)
)

plt.title("HotelIQ - Cancellation Rate by Booking Lead Time")
plt.xlabel("Booking Lead Time")
plt.ylabel("Cancellation Rate (%)")
plt.xticks(rotation=0)
plt.legend(title="Hotel Type")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

# Analyze cancellations by market segment
segment_analysis = (
    df_analysis.groupby("market_segment")
    .agg(
        total_bookings=("is_canceled", "size"),
        canceled_bookings=("is_canceled", "sum"),
        cancellation_rate=("is_canceled", "mean"),
        average_lead_time=("lead_time", "mean"),
        average_adr=("adr", "mean")
    )
    .reset_index()
)

# Convert cancellation rate to percentage
segment_analysis["cancellation_rate"] *= 100

# Sort from highest to lowest cancellation rate
segment_analysis = segment_analysis.sort_values(
    "cancellation_rate",
    ascending=False
)

display(segment_analysis.round(2))

# Visualize cancellation rates
plt.figure(figsize=(11, 6))

plt.barh(
    segment_analysis["market_segment"],
    segment_analysis["cancellation_rate"]
)

plt.title("HotelIQ - Cancellation Rate by Market Segment")
plt.xlabel("Cancellation Rate (%)")
plt.ylabel("Market Segment")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:

import pandas as pd
from scipy.stats import chi2_contingency

# Compare Group and Direct bookings
comparison = df_analysis[
    df_analysis["market_segment"].isin(["Groups", "Direct"])
].copy()

# Build a contingency table
contingency_table = pd.crosstab(
    comparison["market_segment"],
    comparison["is_canceled"]
)

print("Contingency Table:")
display(contingency_table)

# Chi-square test
chi2, p_value, dof, expected = chi2_contingency(
    contingency_table
)

print("Chi-square statistic:", round(chi2, 2))
print("P-value:", p_value)
print("Degrees of freedom:", dof)

alpha = 0.05

if p_value < alpha:
    print("Result: Statistically significant association.")
else:
    print("Result: No statistically significant association detected.")

In [ ]:

# Create a unique technical ID for each row
df_sql = df_analysis.copy().reset_index(drop=True)

df_sql.insert(
    0,
    "booking_id",
    range(1, len(df_sql) + 1)
)

# Check uniqueness
print("Total rows:", len(df_sql))
print("Unique booking IDs:", df_sql["booking_id"].nunique())

display(df_sql.head())

In [ ]:

# Export the data for PostgreSQL
df_sql.to_csv(
    "HotelIQ_SQL_Ready.csv",
    index=False
)

from google.colab import files
files.download("HotelIQ_SQL_Ready.csv")

print("SQL-ready dataset exported successfully!")

In [ ]:

# Create the database import dataset
df_sql = df_analysis.copy().reset_index(drop=True)

# Remove helper column if present
df_sql = df_sql.drop(
    columns=["data_quality_issue"],
    errors="ignore"
)

# Remove generated booking_id if rerunning this cell
df_sql = df_sql.drop(
    columns=["booking_id"],
    errors="ignore"
)

# Add a unique technical identifier
df_sql.insert(0, "booking_id", range(1, len(df_sql) + 1))

# Use the exact column order expected by PostgreSQL
columns = [
    "booking_id", "hotel", "is_canceled", "lead_time",
    "arrival_date_year", "arrival_date_month",
    "arrival_date_week_number", "arrival_date_day_of_month",
    "stays_in_weekend_nights", "stays_in_week_nights",
    "adults", "children", "babies", "meal", "country",
    "market_segment", "distribution_channel",
    "is_repeated_guest", "previous_cancellations",
    "previous_bookings_not_canceled",
    "reserved_room_type", "assigned_room_type",
    "booking_changes", "deposit_type", "agent",
    "company", "days_in_waiting_list", "customer_type",
    "adr", "required_car_parking_spaces",
    "total_of_special_requests", "reservation_status",
    "reservation_status_date", "total_nights",
    "total_guests", "arrival_date",
    "estimated_booking_value", "lead_time_group"
]

df_sql = df_sql[columns]

df_sql.to_csv("HotelIQ_SQL_Ready.csv", index=False)

from google.colab import files
files.download("HotelIQ_SQL_Ready.csv")

print("Rows:", len(df_sql))
print("Columns:", len(df_sql.columns))


In [ ]:

print("Dataset shape:", df_analysis.shape)
print("Cancellation distribution:")
print(df_analysis["is_canceled"].value_counts())


In [ ]:

# Step 17B - Select features and target

features = [
    "hotel",
    "lead_time",
    "market_segment",
    "deposit_type",
    "customer_type",
    "total_nights",
    "adr"
]

# X contains the information used for prediction
X = df_analysis[features].copy()

# y contains the actual cancellation result
y = df_analysis["is_canceled"].copy()

print("Features shape:", X.shape)
print("Target shape:", y.shape)

print("\nFirst 5 rows of features:")
display(X.head())

print("\nMissing values:")
print(X.isnull().sum())


In [ ]:

# Step 17C - Split the dataset

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

print("\nTraining cancellation rate:")
print(round(y_train.mean() * 100, 2), "%")

print("\nTesting cancellation rate:")
print(round(y_test.mean() * 100, 2), "%")


In [ ]:

# Step 17D - Create preprocessing pipeline

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Categorical columns
categorical_features = [
    "hotel",
    "market_segment",
    "deposit_type",
    "customer_type"
]

# Numerical columns
numerical_features = [
    "lead_time",
    "total_nights",
    "adr"
]

# Define preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numerical",
            StandardScaler(),
            numerical_features
        )
    ]
)

print("Preprocessing pipeline created successfully!")
print("\nCategorical features:", categorical_features)
print("Numerical features:", numerical_features)


In [ ]:

# Step 17E - Train HotelIQ Logistic Regression model

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Combine preprocessing and machine learning
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Train using only the training dataset
model.fit(X_train, y_train)

print("HotelIQ Logistic Regression model trained successfully!")
print("Training bookings:", len(X_train))
print("Testing bookings:", len(X_test))


In [ ]:
# Step 17F - Evaluate HotelIQ model

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Predict cancellation outcomes
y_pred = model.predict(X_test)

# Predict cancellation probabilities
y_prob = model.predict_proba(X_test)[:, 1]

# Calculate performance metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("HOTELIQ MODEL PERFORMANCE")
print("-------------------------")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")

print("\nCONFUSION MATRIX")
print(confusion_matrix(y_test, y_pred))

print("\nCLASSIFICATION REPORT")
print(classification_report(y_test, y_pred))


In [ ]:

# Step 17G - Compare model against baseline

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Baseline: predict every booking as not canceled
baseline_pred = np.zeros(len(y_test), dtype=int)

results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score"
    ],
    "Baseline": [
        accuracy_score(y_test, baseline_pred),
        precision_score(
            y_test, baseline_pred, zero_division=0
        ),
        recall_score(y_test, baseline_pred),
        f1_score(y_test, baseline_pred)
    ],
    "Logistic Regression": [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred),
        recall_score(y_test, y_pred),
        f1_score(y_test, y_pred)
    ]
})

print("HOTELIQ MODEL COMPARISON")
display(results.round(4))

In [ ]:

# Step 17H - Compare classification thresholds

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

import pandas as pd

threshold_results = []

for threshold in [0.50, 0.40, 0.30]:

    # Convert probabilities into predictions
    predictions = (y_prob >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(
            y_test, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_test, predictions, zero_division=0
        ),
        "F1-score": f1_score(
            y_test, predictions, zero_division=0
        )
    })

threshold_df = pd.DataFrame(threshold_results)

print("HOTELIQ THRESHOLD COMPARISON")
display(threshold_df.round(4))


In [ ]:
# Step 17I - Train HotelIQ Random Forest

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

# Create a second model using the same preprocessing
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        max_depth=12,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    ))
])

# Train Random Forest on training data only
rf_model.fit(X_train, y_train)

print("HotelIQ Random Forest trained successfully!")
print("Training bookings:", len(X_train))
print("Testing bookings:", len(X_test))


In [ ]:

# Step 17J - Compare HotelIQ ML models

import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# Logistic Regression predictions
lr_pred = model.predict(X_test)
lr_prob = model.predict_proba(X_test)[:, 1]

# Random Forest predictions
rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

# Calculate metrics for both models
comparison = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "ROC-AUC"
    ],
    "Logistic Regression": [
        accuracy_score(y_test, lr_pred),
        precision_score(y_test, lr_pred),
        recall_score(y_test, lr_pred),
        f1_score(y_test, lr_pred),
        roc_auc_score(y_test, lr_prob)
    ],
    "Random Forest": [
        accuracy_score(y_test, rf_pred),
        precision_score(y_test, rf_pred),
        recall_score(y_test, rf_pred),
        f1_score(y_test, rf_pred),
        roc_auc_score(y_test, rf_prob)
    ]
})

print("HOTELIQ MACHINE LEARNING MODEL COMPARISON")
display(comparison.round(4))

print("\nRANDOM FOREST CONFUSION MATRIX")
print(confusion_matrix(y_test, rf_pred))


In [ ]:
# Step 17K - Random Forest feature importance

import pandas as pd
import matplotlib.pyplot as plt

# Retrieve trained preprocessing and classifier
trained_preprocessor = rf_model.named_steps["preprocessor"]
trained_forest = rf_model.named_steps["classifier"]

# Retrieve transformed feature names
feature_names = trained_preprocessor.get_feature_names_out()

# Retrieve Random Forest feature importance
importances = trained_forest.feature_importances_

# Create a feature importance table
feature_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
})

# Sort by importance
feature_importance_df = (
    feature_importance_df
    .sort_values("Importance", ascending=False)
)

print("HOTELIQ TOP 15 PREDICTIVE FEATURES")
display(feature_importance_df.head(15))

# Plot the top 15 features
plt.figure(figsize=(10, 7))

top_features = feature_importance_df.head(15)

plt.barh(
    top_features["Feature"][::-1],
    top_features["Importance"][::-1]
)

plt.xlabel("Feature Importance")
plt.title("HotelIQ - Top 15 Cancellation Prediction Features")
plt.tight_layout()
plt.show()

In [ ]:

# Step 17L - Check train/test feature overlap

import pandas as pd

# Unique feature combinations in training data
train_unique = X_train.drop_duplicates()

# Identify test rows matching training feature combinations
test_with_match = X_test.merge(
    train_unique.assign(found_in_train=True),
    on=list(X_train.columns),
    how="left"
)

matched_rows = test_with_match["found_in_train"].fillna(False).sum()

print("HOTELIQ TRAIN/TEST OVERLAP CHECK")
print("--------------------------------")
print("Training bookings:", len(X_train))
print("Testing bookings:", len(X_test))
print("Test rows matching training features:", int(matched_rows))
print(
    "Percentage of test rows with matching features:",
    round(matched_rows / len(X_test) * 100, 2),
    "%"
)


In [ ]:
# Step 17M - Group-aware train/test split

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# Create a group ID for each unique feature combination
groups = pd.util.hash_pandas_object(
    X.reset_index(drop=True),
    index=False
)

# Split groups rather than individual rows
group_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    group_split.split(X, y, groups=groups)
)

# Create new group-aware datasets
X_train_group = X.iloc[train_idx].copy()
X_test_group = X.iloc[test_idx].copy()

y_train_group = y.iloc[train_idx].copy()
y_test_group = y.iloc[test_idx].copy()

# Check for shared feature groups
train_groups = set(groups.iloc[train_idx])
test_groups = set(groups.iloc[test_idx])

overlap = len(train_groups.intersection(test_groups))

print("HOTELIQ GROUP-AWARE SPLIT")
print("-------------------------")
print("Training bookings:", len(X_train_group))
print("Testing bookings:", len(X_test_group))
print("Overlapping feature groups:", overlap)

print(
    "Training cancellation rate:",
    round(y_train_group.mean() * 100, 2), "%"
)

print(
    "Testing cancellation rate:",
    round(y_test_group.mean() * 100, 2), "%"
)

In [ ]:

# Step 17N - Retrain models using group-aware split

from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
import pandas as pd

# Create independent copies of our existing pipelines
lr_group_model = clone(model)
rf_group_model = clone(rf_model)

# Train only on group-aware training data
lr_group_model.fit(X_train_group, y_train_group)
rf_group_model.fit(X_train_group, y_train_group)

# Evaluate each model on group-aware test data
group_results = []

for name, trained_model in [
    ("Logistic Regression", lr_group_model),
    ("Random Forest", rf_group_model)
]:
    predictions = trained_model.predict(X_test_group)
    probabilities = trained_model.predict_proba(
        X_test_group
    )[:, 1]

    group_results.append({
        "Model": name,
        "Accuracy": accuracy_score(
            y_test_group, predictions
        ),
        "Precision": precision_score(
            y_test_group, predictions,
            zero_division=0
        ),
        "Recall": recall_score(
            y_test_group, predictions,
            zero_division=0
        ),
        "F1-score": f1_score(
            y_test_group, predictions,
            zero_division=0
        ),
        "ROC-AUC": roc_auc_score(
            y_test_group, probabilities
        )
    })

group_comparison = pd.DataFrame(group_results)

print("HOTELIQ GROUP-AWARE MODEL COMPARISON")
display(group_comparison.round(4))

In [ ]:

# Step 17O - Create training and validation sets

from sklearn.model_selection import GroupShuffleSplit
import pandas as pd

# Create groups from the existing training features
validation_groups = pd.util.hash_pandas_object(
    X_train_group.reset_index(drop=True),
    index=False
)

# Split the training data into training and validation
validation_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=123
)

train_sub_idx, val_idx = next(
    validation_split.split(
        X_train_group,
        y_train_group,
        groups=validation_groups
    )
)

# Create the new datasets
X_train_final = X_train_group.iloc[train_sub_idx].copy()
X_val = X_train_group.iloc[val_idx].copy()

y_train_final = y_train_group.iloc[train_sub_idx].copy()
y_val = y_train_group.iloc[val_idx].copy()

# Verify no feature-group overlap
train_ids = set(validation_groups.iloc[train_sub_idx])
val_ids = set(validation_groups.iloc[val_idx])

print("HOTELIQ TRAINING / VALIDATION / TEST SPLIT")
print("-----------------------------------------")
print("Training bookings:", len(X_train_final))
print("Validation bookings:", len(X_val))
print("Testing bookings:", len(X_test_group))
print("Training/validation group overlap:",
      len(train_ids.intersection(val_ids)))

print("\nCancellation rates:")
print("Training:", round(y_train_final.mean() * 100, 2), "%")
print("Validation:", round(y_val.mean() * 100, 2), "%")
print("Testing:", round(y_test_group.mean() * 100, 2), "%")


In [ ]:
# Step 17P - Train Random Forest for threshold selection

from sklearn.base import clone

# Create a fresh copy of the Random Forest pipeline
rf_final = clone(rf_model)

# Train using only the training dataset
rf_final.fit(X_train_final, y_train_final)

# Generate probabilities for validation bookings
val_prob = rf_final.predict_proba(X_val)[:, 1]

print("HOTELIQ RANDOM FOREST VALIDATION MODEL")
print("--------------------------------------")
print("Model trained successfully!")
print("Training bookings:", len(X_train_final))
print("Validation bookings:", len(X_val))
print("Predicted probabilities:", len(val_prob))

print("\nFirst 5 cancellation probabilities:")
print(val_prob[:5].round(4))


In [ ]:

# Step 17Q - Compare Random Forest thresholds

import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Thresholds to compare
thresholds = [0.20, 0.30, 0.40, 0.50, 0.60]

results = []

for threshold in thresholds:

    # Convert probabilities into cancellation predictions
    predictions = (val_prob >= threshold).astype(int)

    # Calculate the confusion matrix
    tn, fp, fn, tp = confusion_matrix(
        y_val, predictions, labels=[0, 1]
    ).ravel()

    results.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_val, predictions),
        "Precision": precision_score(
            y_val, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_val, predictions, zero_division=0
        ),
        "F1-score": f1_score(
            y_val, predictions, zero_division=0
        ),
        "False Alarms": fp,
        "Missed Cancellations": fn
    })

threshold_results = pd.DataFrame(results)

print("HOTELIQ VALIDATION THRESHOLD COMPARISON")
display(threshold_results.round(4))

# Find the threshold with the highest validation F1-score
best_row = threshold_results.loc[
    threshold_results["F1-score"].idxmax()
]

best_threshold = float(best_row["Threshold"])

print(
    "\nBest threshold by validation F1-score:",
    best_threshold
)


In [ ]:
# Step 17R - Final Random Forest test evaluation

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Use the threshold selected from validation data
selected_threshold = best_threshold

# Generate predictions for the held-out test set
test_prob = rf_final.predict_proba(X_test_group)[:, 1]

test_pred = (test_prob >= selected_threshold).astype(int)

print("HOTELIQ FINAL RANDOM FOREST EVALUATION")
print("-------------------------------------")
print("Selected threshold:", selected_threshold)
print("Test bookings:", len(X_test_group))

print("\nPERFORMANCE METRICS")
print("Accuracy:", round(
    accuracy_score(y_test_group, test_pred), 4
))
print("Precision:", round(
    precision_score(y_test_group, test_pred), 4
))
print("Recall:", round(
    recall_score(y_test_group, test_pred), 4
))
print("F1-score:", round(
    f1_score(y_test_group, test_pred), 4
))
print("ROC-AUC:", round(
    roc_auc_score(y_test_group, test_prob), 4
))

print("\nCONFUSION MATRIX")
print(confusion_matrix(y_test_group, test_pred))

print("\nCLASSIFICATION REPORT")
print(classification_report(y_test_group, test_pred))

In [ ]:

# Step 17S - Save final HotelIQ model metrics

import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

tn, fp, fn, tp = confusion_matrix(
    y_test_group, test_pred, labels=[0, 1]
).ravel()

final_metrics = pd.DataFrame([{
    "Model": "Random Forest",
    "Threshold": selected_threshold,
    "Test_Bookings": len(y_test_group),
    "Accuracy": accuracy_score(y_test_group, test_pred),
    "Precision": precision_score(y_test_group, test_pred),
    "Recall": recall_score(y_test_group, test_pred),
    "F1_Score": f1_score(y_test_group, test_pred),
    "ROC_AUC": roc_auc_score(y_test_group, test_prob),
    "True_Negatives": tn,
    "False_Positives": fp,
    "False_Negatives": fn,
    "True_Positives": tp
}])

# Save to CSV
final_metrics.to_csv(
    "HotelIQ_Final_Model_Metrics.csv",
    index=False
)

print("HotelIQ final model metrics saved successfully!")
display(final_metrics.round(4))

In [ ]:
import pandas as pd
from google.colab import files

# Recreate the final HotelIQ machine-learning results
final_metrics = pd.DataFrame([{
    "Model": "Random Forest",
    "Threshold": 0.40,
    "Test_Bookings": 24071,
    "Accuracy": 0.7490,
    "Precision": 0.6512,
    "Recall": 0.7326,
    "F1_Score": 0.6895,
    "ROC_AUC": 0.8321,
    "True_Negatives": 11317,
    "False_Positives": 3594,
    "False_Negatives": 2449,
    "True_Positives": 6711
}])

filename = "HotelIQ_Final_Model_Metrics.csv"

final_metrics.to_csv(filename, index=False)

print("HotelIQ ML metrics CSV created successfully!")
display(final_metrics)

# Download the file directly to your computer
files.download(filename)


In [ ]:
import pandas as pd
from google.colab import files

feature_importance = pd.DataFrame({
    "Feature": [
        "Non-refundable deposit",
        "No deposit",
        "Lead time",
        "Average daily rate",
        "Online travel agency",
        "Groups",
        "Total nights",
        "Transient customer",
        "Transient-party customer",
        "Direct booking"
    ],
    "Importance": [
        0.316720,
        0.189514,
        0.183171,
        0.071202,
        0.043741,
        0.035940,
        0.034102,
        0.029244,
        0.026390,
        0.021182
    ]
})

filename = "HotelIQ_Feature_Importance.csv"

feature_importance.to_csv(filename, index=False)

display(feature_importance)

files.download(filename)


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.base import clone

# Create a separate copy of the existing pipeline
rf_tuning = clone(rf_model)

# Small hyperparameter search
param_distributions = {
    "classifier__n_estimators": [100, 150],
    "classifier__max_depth": [8, 12, 16],
    "classifier__min_samples_leaf": [3, 5, 10]
}

# Tune using training data only
random_search = RandomizedSearchCV(
    estimator=rf_tuning,
    param_distributions=param_distributions,
    n_iter=5,
    scoring="roc_auc",
    cv=3,
    random_state=42,
    n_jobs=1,
    verbose=1
)

random_search.fit(X_train_final, y_train_final)

print("Best parameters:", random_search.best_params_)
print("Best cross-validation ROC-AUC:",
      round(random_search.best_score_, 4))

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.base import clone

# Create a separate copy of the existing pipeline
rf_tuning = clone(rf_model)

# Small hyperparameter search
param_distributions = {
    "classifier__n_estimators": [100, 150],
    "classifier__max_depth": [8, 12, 16],
    "classifier__min_samples_leaf": [3, 5, 10]
}

# Tune using training data only
random_search = RandomizedSearchCV(
    estimator=rf_tuning,
    param_distributions=param_distributions,
    n_iter=5,
    scoring="roc_auc",
    cv=3,
    random_state=42,
    n_jobs=1,
    verbose=1
)

random_search.fit(X_train_final, y_train_final)

print("Best parameters:", random_search.best_params_)
print("Best cross-validation ROC-AUC:",
      round(random_search.best_score_, 4))